# NB03 - Synthetic Medical Claims
### Cross-Benefit Duplicate Drug Billing Detection - Medical Leg

**Schema:** `cross_benefit_audit`<br>
**Tables created:** `medical_claims`<br>
**Depends on:** NB01 (`target_hcpcs_crosswalk`), NB02 (`seeded_duplicate_events`, `member_registry`)

---

## What this notebook builds

Medical benefit drug claims travel on **CMS-1500** (professional) or **UB-04** (institutional) forms and identify drugs with **HCPCS J/Q codes** - not NDCs. The drug itself is purchased by the provider under a buy-and-bill arrangement: the provider acquires the drug, administers it, and bills the payer for both the drug (J/Q code) and the administration (96413/96415 or similar CPT codes)

The pharmacy benefit plays no role in a clean buy-and-bill claim.

Cross-benefit duplicate billing arises when the drug also appears on a pharmacy claim for the same member and overlapping service window - both legs billing for the same molecule.

### Seeded medical claims

Each seeded medical claim is the counterpart of a seeded pharmacy claim from NB02.<br>
NB02 wrote the HCPCS code, date offset, shared prescriber NPI, and administered units for each scenario into `seeded_duplicate_events`.<br>
NB03 reads that registry and generates the medical leg deterministically.

| Scenario | Medical HCPCS | Rendering NPI | hcpcs_units |
|---|---|---|---|
| EXACT_SAME_CODE | same as pharmacy | = pharmacy prescriber_npi | adm_units ± 8% noise |
| NEAR_DATE | same as pharmacy | = pharmacy prescriber_npi | adm_units ± 8% noise |
| QCODE_GAP | originator J-code | = pharmacy prescriber_npi | adm_units ± 8% noise |
| NEW_FORM_CROSS | IV J-code | = pharmacy prescriber_npi | adm_units ± 8% noise |
| DIFF_BIOSIM | different Q-code | = pharmacy prescriber_npi | adm_units ± 8% noise |

## 0 - Setup

In [0]:
## 0 - Setup

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
from datetime import date, timedelta
import random, hashlib

spark = SparkSession.builder.getOrCreate()

RANDOM_SEED  = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

PERIOD_START = date(2026, 10, 1)
PERIOD_END   = date(2026, 12, 31)

CROSS_WALK_TABLE  = "cross_benefit_audit.target_hcpcs_crosswalk"
MEMBER_TABLE = "cross_benefit_audit.member_registry"
SEED_TABLE   = "cross_benefit_audit.seeded_duplicate_events"
MED_TABLE    = "cross_benefit_audit.medical_claims"

N_CLEAN_MED  = 680 # clean (non-duplicate) medical claims

print("Setup complete.")
print(f"Period : {PERIOD_START} --> {PERIOD_END}")
print(f"Clean medical claims target: {N_CLEAN_MED}")

Setup complete.
Period : 2026-10-01 --> 2026-12-31
Clean medical claims target: 680


---
## 1 - Load reference tables

In [0]:
df_cross_walk  = spark.table(CROSS_WALK_TABLE).toPandas()
df_members = spark.table(MEMBER_TABLE).toPandas()
df_seeds  = spark.table(SEED_TABLE).toPandas()

# rx_service_date arrives as a date/datetime column from NB02
df_seeds["rx_service_date"] = pd.to_datetime(df_seeds["rx_service_date"]).dt.date

print(f"Crosswalk rows : {len(df_cross_walk)}")
print(f"Members        : {len(df_members)}")
print(f"Seeded events  : {len(df_seeds)}")
print()
print("Seeded event schema:")
print(df_seeds.dtypes.to_string())

Crosswalk rows : 31
Members        : 200
Seeded events  : 72

Seeded event schema:
seed_event_id                object
scenario_type                object
member_id                    object
molecule                     object
rx_claim_id                  object
rx_ndc                       object
rx_hcpcs_code                object
rx_service_date              object
expected_med_hcpcs           object
med_date_offset               int64
shared_prescriber_npi        object
administered_hcpcs_units    float64
description                  object


## 2 - HCPCS drug parameters

The medical benefit identifies the drug by HCPCS code, not NDC.<br>
`units_range` reflects standard administered dose in billing units.<br>
`asp_per_unit` is the approximate CMS ASP reimbursement rate per billing unit.

In [0]:
HCPCS_PARAMS = {
    # Ophthalmology -- professional, place of service 11 (office/clinic)
    "J2778": {"molecule": "ranibizumab", "drug_type": "originator",
               "units_range": (3, 5), "asp_per_unit": 28.50,
               "claim_type": "professional", "pos": "11", "revenue_code": None,
               "icd10_pool": ["H35.32", "H35.31", "H35.354", "H34.8113"]},
    "Q5124": {"molecule": "ranibizumab", "drug_type": "biosimilar",
               "units_range": (5, 5), "asp_per_unit": 17.20,
               "claim_type": "professional", "pos": "11", "revenue_code": None,
               "icd10_pool": ["H35.32", "H35.31", "H35.354"]},
    "Q5128": {"molecule": "ranibizumab", "drug_type": "biosimilar",
               "units_range": (3, 5), "asp_per_unit": 16.80,
               "claim_type": "professional", "pos": "11", "revenue_code": None,
               "icd10_pool": ["H35.32", "H35.31", "H35.354"]},
    "J0178": {"molecule": "aflibercept", "drug_type": "originator",
               "units_range": (2, 2), "asp_per_unit": 540.00,
               "claim_type": "professional", "pos": "11", "revenue_code": None,
               "icd10_pool": ["H35.32", "E11.319", "H35.354", "H34.8113"]},
    "Q5138": {"molecule": "aflibercept", "drug_type": "biosimilar",
               "units_range": (2, 2), "asp_per_unit": 380.00,
               "claim_type": "professional", "pos": "11", "revenue_code": None,
               "icd10_pool": ["H35.32", "E11.319", "H35.354"]},
    # Oncology and autoimmune -- institutional, place of service 22 (outpatient hospital)
    "J9035": {"molecule": "bevacizumab", "drug_type": "originator",
               "units_range": (40, 70), "asp_per_unit": 12.60,
               "claim_type": "institutional", "pos": "22", "revenue_code": "0636",
               "icd10_pool": ["C18.9", "C34.90", "C56.9", "C64.9"]},
    "Q5107": {"molecule": "bevacizumab", "drug_type": "biosimilar",
               "units_range": (40, 70), "asp_per_unit": 9.80,
               "claim_type": "institutional", "pos": "22", "revenue_code": "0636",
               "icd10_pool": ["C18.9", "C34.90", "C56.9", "C64.9"]},
    "Q5118": {"molecule": "bevacizumab", "drug_type": "biosimilar",
               "units_range": (40, 70), "asp_per_unit": 9.40,
               "claim_type": "institutional", "pos": "22", "revenue_code": "0636",
               "icd10_pool": ["C18.9", "C34.90", "C56.9"]},
    "J9271": {"molecule": "pembrolizumab", "drug_type": "originator",
               "units_range": (200, 200), "asp_per_unit": 47.80,
               "claim_type": "institutional", "pos": "22", "revenue_code": "0636",
               "icd10_pool": ["C43.9", "C34.90", "C67.9", "C18.9", "C80.1"]},
    "J9277": {"molecule": "pembrolizumab", "drug_type": "new_formulation",
               "units_range": (400, 400), "asp_per_unit": 23.90,
               "claim_type": "institutional", "pos": "22", "revenue_code": "0636",
               "icd10_pool": ["C43.9", "C34.90", "C67.9", "C18.9"]},
    "J1745": {"molecule": "infliximab", "drug_type": "originator",
               "units_range": (30, 45), "asp_per_unit": 83.20,
               "claim_type": "institutional", "pos": "22", "revenue_code": "0636",
               "icd10_pool": ["M05.79", "K50.90", "K51.90", "M45.9", "L40.59"]},
    "Q5103": {"molecule": "infliximab", "drug_type": "biosimilar",
               "units_range": (30, 45), "asp_per_unit": 46.10,
               "claim_type": "institutional", "pos": "22", "revenue_code": "0636",
               "icd10_pool": ["M05.79", "K50.90", "K51.90", "M45.9"]},
    "Q5104": {"molecule": "infliximab", "drug_type": "biosimilar",
               "units_range": (30, 45), "asp_per_unit": 44.80,
               "claim_type": "institutional", "pos": "22", "revenue_code": "0636",
               "icd10_pool": ["M05.79", "K50.90", "K51.90", "M45.9"]},
    "Q5121": {"molecule": "infliximab", "drug_type": "biosimilar",
               "units_range": (30, 45), "asp_per_unit": 43.50,
               "claim_type": "institutional", "pos": "22", "revenue_code": "0636",
               "icd10_pool": ["M05.79", "K50.90", "K51.90", "M45.9"]},
    "J2350": {"molecule": "ocrelizumab", "drug_type": "originator",
               "units_range": (600, 600), "asp_per_unit": 17.40,
               "claim_type": "institutional", "pos": "22", "revenue_code": "0636",
               "icd10_pool": ["G35"]},
    "J2351": {"molecule": "ocrelizumab", "drug_type": "new_formulation",
               "units_range": (920, 920), "asp_per_unit": 11.20,
               "claim_type": "institutional", "pos": "22", "revenue_code": "0636",
               "icd10_pool": ["G35"]},
}

# Validate - every HCPCS in seeded events must be in HCPCS_PARAMS
for code in df_seeds["expected_med_hcpcs"].unique():
    assert code in HCPCS_PARAMS, f"Missing HCPCS_PARAMS entry for: {code}"

print(f"HCPCS codes defined: {len(HCPCS_PARAMS)}")

HCPCS codes defined: 16


## 3 - Provider and facility pools

Four provider specialties align to the four therapeutic areas in our drug portfolio.<br>
NPI seeds are distinct from NB02 pharmacy NPI seeds (1001--1008).

In [0]:
def make_npi(seed_int: int) -> str:
    """10-digit NPI, starts with 1 (common for individuals)."""
    rng = random.Random(seed_int)
    digits = "".join(str(rng.randint(0, 9)) for _ in range(9))
    return f"1{digits}"

def make_med_claim_id(service_date: date, seq: int) -> str:
    return f"MED-{service_date.strftime('%Y%m%d')}-{seq:05d}"

def random_date_in_period(rng, start: date = PERIOD_START, end: date = PERIOD_END) -> date:
    delta = (end - start).days
    return start + timedelta(days=rng.randint(0, delta))

# Provider pool by specialty
# Seeds 2001-2005 ophthalmology, 2011-2015 oncology, 2021-2024 autoimmune, 2031-2033 neurology
PROVIDERS = {
    "ophthalmology": [
        {"name": "Dr. Howard Bellman",   "npi": make_npi(2001), "specialty": "ophthalmology"},
        {"name": "Dr. Patricia Suen",    "npi": make_npi(2002), "specialty": "ophthalmology"},
        {"name": "Dr. George Andreou",   "npi": make_npi(2003), "specialty": "ophthalmology"},
        {"name": "Dr. Celine Marchetti", "npi": make_npi(2004), "specialty": "ophthalmology"},
        {"name": "Dr. Jameel Okonkwo",   "npi": make_npi(2005), "specialty": "ophthalmology"},
    ],
    "oncology": [
        {"name": "Dr. Sandra Herrera",   "npi": make_npi(2011), "specialty": "oncology"},
        {"name": "Dr. Warren Holt",      "npi": make_npi(2012), "specialty": "oncology"},
        {"name": "Dr. Nalini Chandra",   "npi": make_npi(2013), "specialty": "oncology"},
        {"name": "Dr. Marcus Webb",      "npi": make_npi(2014), "specialty": "oncology"},
        {"name": "Dr. Irene Tsoukalas",  "npi": make_npi(2015), "specialty": "oncology"},
    ],
    "autoimmune": [
        {"name": "Dr. Daniel Kramer",    "npi": make_npi(2021), "specialty": "rheumatology"},
        {"name": "Dr. Yuki Fujimori",    "npi": make_npi(2022), "specialty": "rheumatology"},
        {"name": "Dr. Abena Asante",     "npi": make_npi(2023), "specialty": "gastroenterology"},
        {"name": "Dr. Filip Novotny",    "npi": make_npi(2024), "specialty": "rheumatology"},
    ],
    "neurology": [
        {"name": "Dr. Elena Volkov",     "npi": make_npi(2031), "specialty": "neurology"},
        {"name": "Dr. Simon Tran",       "npi": make_npi(2032), "specialty": "neurology"},
        {"name": "Dr. Constance Okoro",  "npi": make_npi(2033), "specialty": "neurology"},
    ],
}

# Outpatient infusion / institutional facilities
# Seeds 3001-3008
FACILITIES = [
    {"name": "Memorial Infusion Center",        "npi": make_npi(3001), "state": "TN"},
    {"name": "Suncoast Oncology Institute",      "npi": make_npi(3002), "state": "FL"},
    {"name": "Pacific Infusion Services",        "npi": make_npi(3003), "state": "CA"},
    {"name": "Great Plains Cancer Center",       "npi": make_npi(3004), "state": "TX"},
    {"name": "Lakeside Infusion Clinic",         "npi": make_npi(3005), "state": "IL"},
    {"name": "Northeast Specialty Infusion",     "npi": make_npi(3006), "state": "NY"},
    {"name": "Cascade Rheumatology Center",      "npi": make_npi(3007), "state": "WA"},
    {"name": "Heartland Medical Group",          "npi": make_npi(3008), "state": "OH"},
]

# Therapeutic area --> provider specialty mapping
# Drives which PROVIDERS sub-pool gets used for a given drug
TA_TO_SPECIALTY = {
    "ophthalmology": "ophthalmology",
    "oncology":      "oncology",
    "autoimmune":    "autoimmune",
    "neurology":     "neurology",
}

# Buy-and-bill provider pool.
# These 5 providers appear as rendering_npi on seeded medical claims
# and as prescriber_npi on the corresponding seeded pharmacy claims (NB02 wrote
# shared_prescriber_npi into seeded_duplicate_events).
# A deterministic 20% of members are also flagged buy-and-bill eligible; their
# clean claims draw from this pool too, producing realistic FP NPI co-location.
# Seeds 9001-9005 are distinct from all other provider/pharmacy seeds.
BUY_AND_BILL_PROVIDERS = [
    {"name": "Dr. Victor Osei",     "npi": make_npi(9001)},
    {"name": "Dr. Tamara Voronova", "npi": make_npi(9002)},
    {"name": "Dr. Alan Finch",      "npi": make_npi(9003)},
    {"name": "Dr. Soo-Jin Park",    "npi": make_npi(9004)},
    {"name": "Dr. Rafael Iglesias", "npi": make_npi(9005)},
]

def get_buy_and_bill_npi(member_id: str) -> str:
    """Deterministic buy-and-bill NPI for a member.
    Hashes member_id to one of the 5 buy-and-bill providers.
    NB02 uses the identical function so both notebooks assign the same NPI
    for the same member without sharing state."""
    idx = int(hashlib.md5(member_id.encode()).hexdigest(), 16) % len(BUY_AND_BILL_PROVIDERS)
    return BUY_AND_BILL_PROVIDERS[idx]["npi"]

def is_buy_and_bill_member(member_id: str) -> bool:
    """20% of members are buy-and-bill eligible (deterministic).
    NB02 uses the identical function for consistent NPI assignment across notebooks."""
    return int(hashlib.md5(member_id.encode()).hexdigest(), 16) % 5 == 0

def pick_provider(therapeutic_area: str, rng) -> dict:
    specialty = TA_TO_SPECIALTY.get(therapeutic_area, "oncology")
    return rng.choice(PROVIDERS[specialty])

def pick_facility(rng) -> dict:
    return rng.choice(FACILITIES)

print(f"PROVIDERS defined:")
for spec, pool in PROVIDERS.items():
    print(f"  {spec:<16}: {len(pool)} providers")
print(f"FACILITIES defined: {len(FACILITIES)}")
print(f"Buy-and-bill providers: {len(BUY_AND_BILL_PROVIDERS)}")

PROVIDERS defined:
  ophthalmology   : 5 providers
  oncology        : 5 providers
  autoimmune      : 4 providers
  neurology       : 3 providers
FACILITIES defined: 8
Buy-and-bill providers: 5


---
## 4 - Medical claim builder

In [0]:
def build_med_claim(seq: int, member_id: str, hcpcs_code: str,
                    service_date: date, rng,
                    is_seeded: bool = False,
                    seed_event_id: str = None,
                    seed_scenario: str = None,
                    override_rendering_npi: str = None,
                    override_hcpcs_units: int = None) -> dict:
    """Assemble one medical (CMS-1500 / UB-04) drug claim record.

    override_rendering_npi: when provided, bypasses the per-claim provider pool draw and
        writes the shared buy-and-bill NPI from seeded_duplicate_events into rendering_npi.
        This is what the NPI co-location check in NB05 fires on: rendering_npi == prescriber_npi.
        For buy-and-bill-eligible clean claims, the eligibility check below handles those
        consistently; for all other clean claims the pool draw stands.
    override_hcpcs_units: when provided, replaces the random randint from units_range with
        the administered_hcpcs_units (±8% noise applied by caller) that NB02 derived and
        stored in seeded_duplicate_events. This produces concordance_ratio ≈ 1.0 for TPs.
        Only passed for seeded claims; clean claims retain the random draw from units_range."""
    p = HCPCS_PARAMS[hcpcs_code]

    # units: use override for seeded claims (dose concordance with pharmacy quantity_dispensed);
    # draw from units_range for clean claims.
    if override_hcpcs_units is not None:
        units = int(override_hcpcs_units)
    else:
        units = rng.randint(*p["units_range"])

    asp_plus_6   = p["asp_per_unit"] * 1.06
    charge_ratio = rng.uniform(1.15, 1.40)
    billed       = round(asp_plus_6 * charge_ratio * units, 2)
    allowed      = round(asp_plus_6 * units, 2)
    copay        = round(rng.uniform(0, min(50.0, allowed * 0.10)), 2)
    plan_paid    = round(allowed - copay, 2)

    xrow = df_cross_walk[df_cross_walk["HCPCS_CODE"] == hcpcs_code]
    ta   = xrow["therapeutic_area"].iloc[0] if not xrow.empty else "oncology"
    mol  = p["molecule"]
    dt   = p["drug_type"]
    icd_pri = rng.choice(p["icd10_pool"])
    icd_sec = rng.choice(p["icd10_pool"]) if rng.random() < 0.5 else None

    # Always call pick_provider for name and specialty display fields; the npi field from
    # the pool is only used as fallback when no override applies and the member is not
    # buy-and-bill eligible. This preserves the rng sequence for all downstream draws
    # (facility, paid_date) regardless of which NPI branch is taken.
    prov       = pick_provider(ta, rng)
    claim_type = p["claim_type"]
    facility_npi = pick_facility(rng)["npi"] if claim_type == "institutional" else None
    billing_npi  = facility_npi if facility_npi else prov["npi"]

    # Rendering NPI: seeded claims use the shared buy-and-bill NPI from seeded_duplicate_events
    # (coordinated with NB02 prescriber_npi); buy-and-bill-eligible members' clean claims draw
    # from the same pool deterministically so NB05 surfaces realistic FP NPI co-location;
    # all other clean claims use the specialty-matched provider pool draw.
    if override_rendering_npi is not None:
        actual_rendering_npi = override_rendering_npi
    elif is_buy_and_bill_member(member_id):
        actual_rendering_npi = get_buy_and_bill_npi(member_id)
    else:
        actual_rendering_npi = prov["npi"]

    return {
        "med_claim_id":          make_med_claim_id(service_date, seq),
        "member_id":             member_id,
        "service_date":          str(service_date),
        "paid_date":             str(service_date + timedelta(days=rng.randint(7, 21))),
        "claim_type":            claim_type,
        "place_of_service":      p["pos"],
        "revenue_code":          p["revenue_code"],
        "rendering_npi":         actual_rendering_npi,
        "rendering_name":        prov["name"],
        "rendering_specialty":   prov["specialty"],
        "billing_npi":           billing_npi,
        "facility_npi":          facility_npi,
        "hcpcs_code":            hcpcs_code,
        "hcpcs_units":           float(units),
        "icd10_primary":         icd_pri,
        "icd10_secondary":       icd_sec,
        "billed_amount":         billed,
        "allowed_amount":        allowed,
        "plan_paid":             plan_paid,
        "member_liability":      copay,
        "molecule":              mol,
        "drug_type":             dt,
        "therapeutic_area":      ta,
        "is_seeded_duplicate":   is_seeded,
        "seed_event_id":         seed_event_id,
        "seed_scenario_type":    seed_scenario,
    }

print("build_med_claim() defined.")

build_med_claim() defined.


---
## 5 - Generate seeded medical claims (72 claims)

One claim per row in `seeded_duplicate_events`.<br>
`administered_hcpcs_units` (stored by NB02) drives hcpcs_units with ±8% noise.<br>
`shared_prescriber_npi` (stored by NB02) drives rendering_npi exactly (no noise).

In [0]:
rng_seed    = random.Random(RANDOM_SEED + 301)
seeded_med  = []
med_seq     = 1

# Separate RNG for administered dose derivation.
# Using a distinct seed (RANDOM_SEED+501, matching the +500 offset convention from NB02)
# keeps this RNG isolated from rng_seed so that charge_ratio, ICD codes, and provider
# draws in build_med_claim() are unaffected by the dose override branch.
rng_adm_med = random.Random(RANDOM_SEED + 501)

for _, evt in df_seeds.iterrows():
    svc_date = evt["rx_service_date"] + timedelta(days=int(evt["med_date_offset"]))
    svc_date = min(svc_date, PERIOD_END)

    # Dose concordance: apply independent ±8% noise to the administered units NB02 stored.
    # This simulates normal billing variation (rounding, split administrations) while keeping
    # concordance_ratio = (rx_qty / median_billunits) / med_units close to 1.0 for TPs.
    # max(1, ...) guards against rounding a small unit count to 0.
    raw_adm_units = float(evt["administered_hcpcs_units"])
    med_adm_units = max(1, int(round(raw_adm_units * rng_adm_med.uniform(0.92, 1.08))))

    row = build_med_claim(
        seq=med_seq,
        member_id=evt["member_id"],
        hcpcs_code=evt["expected_med_hcpcs"],
        service_date=svc_date,
        rng=rng_seed,
        is_seeded=True,
        seed_event_id=evt["seed_event_id"],
        seed_scenario=evt["scenario_type"],
        override_rendering_npi=evt["shared_prescriber_npi"],
        override_hcpcs_units=med_adm_units,
    )
    seeded_med.append(row)
    med_seq += 1

df_seeded_med = pd.DataFrame(seeded_med)
print(f"Seeded medical claims generated: {len(df_seeded_med)}")
print()
print("Seeded scenario breakdown:")
print(df_seeded_med["seed_scenario_type"].value_counts().to_string())
print()
print("Avg hcpcs_units by scenario (dose concordance check):")
print(
    df_seeded_med.groupby("seed_scenario_type")["hcpcs_units"]
    .mean().round(1).to_string()
)

Seeded medical claims generated: 72

Seeded scenario breakdown:
seed_scenario_type
EXACT_SAME_CODE    20
QCODE_GAP          18
NEAR_DATE          16
NEW_FORM_CROSS     10
DIFF_BIOSIM         8

Avg hcpcs_units by scenario (dose concordance check):
seed_scenario_type
DIFF_BIOSIM         32.9
EXACT_SAME_CODE    169.4
NEAR_DATE          241.4
NEW_FORM_CROSS     276.5
QCODE_GAP           25.4


---
## 6 - Generate clean medical claims (~680 claims)

These claims have no pharmacy benefit counterpart. They represent legitimate infusion
center encounters where the drug was acquired via buy-and-bill (not through a PBM fill).

In [0]:
rng_clean = random.Random(RANDOM_SEED + 300)

# Build a lookup from molecule --> list of HCPCS codes we can generate
mol_to_codes = {}
for code, p in HCPCS_PARAMS.items():
    mol_to_codes.setdefault(p["molecule"], []).append(code)

# Weight toward originators in clean population (buy-and-bill favors originator J-codes)
clean_med = []

for i in range(N_CLEAN_MED):
    mbr = df_members.sample(1, random_state=rng_clean.randint(0, 999999)).iloc[0]
    mol = mbr["primary_molecule"]

    # Pull available codes; weight toward originator
    codes = mol_to_codes.get(mol, [])
    if not codes:
        mol  = rng_clean.choice(list(mol_to_codes.keys()))
        codes = mol_to_codes[mol]

    # Weight: originator codes get 3x weight vs biosimilar/new_formulation
    weights = [3 if HCPCS_PARAMS[c]["drug_type"] == "originator" else 1 for c in codes]
    hcpcs_code = rng_clean.choices(codes, weights=weights)[0]

    svc_date = random_date_in_period(rng_clean)
    row = build_med_claim(
        seq=med_seq,
        member_id=mbr["member_id"],
        hcpcs_code=hcpcs_code,
        service_date=svc_date,
        rng=rng_clean,
        is_seeded=False,
    )
    clean_med.append(row)
    med_seq += 1

df_clean_med = pd.DataFrame(clean_med)
print(f"Clean medical claims generated: {len(df_clean_med)}")
print()
print("HCPCS code distribution:")
print(df_clean_med["hcpcs_code"].value_counts().to_string())

Clean medical claims generated: 680

HCPCS code distribution:
hcpcs_code
J9035    120
J2350     76
J0178     70
J2778     70
J9271     68
J1745     41
Q5118     32
Q5107     30
J9277     27
Q5124     27
Q5121     24
J2351     23
Q5103     22
Q5138     19
Q5104     16
Q5128     15


---
## 7 - Combine, finalize, and write

In [0]:
df_med = pd.concat([df_seeded_med, df_clean_med], ignore_index=True)

df_med["service_date"] = pd.to_datetime(df_med["service_date"]).dt.date
df_med["paid_date"]    = pd.to_datetime(df_med["paid_date"]).dt.date
df_med["is_seeded_duplicate"] = df_med["is_seeded_duplicate"].astype(bool)

MED_COLS = [
    "med_claim_id", "member_id", "service_date", "paid_date",
    "claim_type", "place_of_service", "revenue_code",
    "rendering_npi", "rendering_name", "rendering_specialty",
    "billing_npi", "facility_npi",
    "hcpcs_code", "hcpcs_units",
    "icd10_primary", "icd10_secondary",
    "billed_amount", "allowed_amount", "plan_paid", "member_liability",
    "molecule", "drug_type", "therapeutic_area",
    "is_seeded_duplicate", "seed_event_id", "seed_scenario_type",
]
df_med = df_med[MED_COLS]

sdf_med = spark.createDataFrame(df_med)

(
    sdf_med.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(MED_TABLE)
)

print(f"Written: {MED_TABLE}")
print(f"Row count: {spark.table(MED_TABLE).count():,}")

Written: cross_benefit_audit.medical_claims
Row count: 752


---
## 8 - Sanity checks

In [0]:
print(f"=== {MED_TABLE} schema ===")
spark.table(MED_TABLE).printSchema()

=== cross_benefit_audit.medical_claims schema ===
root
 |-- med_claim_id: string (nullable = true)
 |-- member_id: string (nullable = true)
 |-- service_date: date (nullable = true)
 |-- paid_date: date (nullable = true)
 |-- claim_type: string (nullable = true)
 |-- place_of_service: string (nullable = true)
 |-- revenue_code: string (nullable = true)
 |-- rendering_npi: string (nullable = true)
 |-- rendering_name: string (nullable = true)
 |-- rendering_specialty: string (nullable = true)
 |-- billing_npi: string (nullable = true)
 |-- facility_npi: string (nullable = true)
 |-- hcpcs_code: string (nullable = true)
 |-- hcpcs_units: double (nullable = true)
 |-- icd10_primary: string (nullable = true)
 |-- icd10_secondary: string (nullable = true)
 |-- billed_amount: double (nullable = true)
 |-- allowed_amount: double (nullable = true)
 |-- plan_paid: double (nullable = true)
 |-- member_liability: double (nullable = true)
 |-- molecule: string (nullable = true)
 |-- drug_type: str

In [0]:
# Seeded vs clean split
print("Seeded vs clean medical claims:")
spark.table(MED_TABLE).groupBy("is_seeded_duplicate").count().show()

Seeded vs clean medical claims:
+-------------------+-----+
|is_seeded_duplicate|count|
+-------------------+-----+
|               true|   72|
|              false|  680|
+-------------------+-----+



In [0]:
# Verify NPI alignment between seeded rx and seeded med
# Every seeded medical claim's rendering_npi should equal the shared_prescriber_npi
# that NB02 stored in seeded_duplicate_events.
med_seeded = spark.table(MED_TABLE).filter("is_seeded_duplicate = true").select(
    "seed_event_id", "rendering_npi"
)
seeds_ref  = spark.table(SEED_TABLE).select(
    "seed_event_id", "shared_prescriber_npi"
)
npi_check = (
    med_seeded
    .join(seeds_ref, "seed_event_id")
    .withColumn("npi_matches",
        F.col("rendering_npi") == F.col("shared_prescriber_npi"))
)
mismatches = npi_check.filter(F.col("npi_matches") == False).count()
total_seeded = npi_check.count()
print(f"Seeded NPI alignment check:")
print(f"  Total seeded medical claims    : {total_seeded}")
print(f"  rendering_npi == shared_npi    : {total_seeded - mismatches}")
print(f"  Mismatches                     : {mismatches}")   # should be 0

Seeded NPI alignment check:
  Total seeded medical claims    : 72
  rendering_npi == shared_npi    : 72
  Mismatches                     : 0


In [0]:
# Dose concordance preview - raw ratio before NB05 crosswalk join
# Expected: seeded ratio (adm_units_med / adm_units_rx) close to 1.0
# This cell shows hcpcs_units distribution per scenario to validate the override path worked.

print("hcpcs_units stats by scenario_type (seeded medical claims):")
(
    spark.table(MED_TABLE)
    .filter("is_seeded_duplicate = true")
    .groupBy("seed_scenario_type")
    .agg(
        F.round(F.avg("hcpcs_units"), 2).alias("avg_units"),
        F.min("hcpcs_units").alias("min_units"),
        F.max("hcpcs_units").alias("max_units"),
        F.count("med_claim_id").alias("n_claims"),
    )
    .orderBy("seed_scenario_type")
    .show(truncate=False)
)

hcpcs_units stats by scenario_type (seeded medical claims):
+------------------+---------+---------+---------+--------+
|seed_scenario_type|avg_units|min_units|max_units|n_claims|
+------------------+---------+---------+---------+--------+
|DIFF_BIOSIM       |32.88    |3.0      |72.0     |8       |
|EXACT_SAME_CODE   |169.45   |2.0      |642.0    |20      |
|NEAR_DATE         |241.44   |2.0      |954.0    |16      |
|NEW_FORM_CROSS    |276.5    |185.0    |633.0    |10      |
|QCODE_GAP         |25.39    |2.0      |63.0     |18      |
+------------------+---------+---------+---------+--------+



In [0]:
# Unit distribution for clean claims (should span the full units_range per HCPCS code)
print("hcpcs_units avg for clean claims by HCPCS code:")
(
    spark.table(MED_TABLE)
    .filter("is_seeded_duplicate = false")
    .groupBy("hcpcs_code", "molecule")
    .agg(
        F.round(F.avg("hcpcs_units"), 2).alias("avg_units"),
        F.count("med_claim_id").alias("n_claims"),
    )
    .orderBy("molecule", "hcpcs_code")
    .show(20, truncate=False)
)

hcpcs_units avg for clean claims by HCPCS code:
+----------+-------------+---------+--------+
|hcpcs_code|molecule     |avg_units|n_claims|
+----------+-------------+---------+--------+
|J0178     |aflibercept  |2.0      |70      |
|Q5138     |aflibercept  |2.0      |19      |
|J9035     |bevacizumab  |53.53    |120     |
|Q5107     |bevacizumab  |57.2     |30      |
|Q5118     |bevacizumab  |54.16    |32      |
|J1745     |infliximab   |37.98    |41      |
|Q5103     |infliximab   |35.95    |22      |
|Q5104     |infliximab   |36.31    |16      |
|Q5121     |infliximab   |36.08    |24      |
|J2350     |ocrelizumab  |600.0    |76      |
|J2351     |ocrelizumab  |920.0    |23      |
|J9271     |pembrolizumab|200.0    |68      |
|J9277     |pembrolizumab|400.0    |27      |
|J2778     |ranibizumab  |3.91     |70      |
|Q5124     |ranibizumab  |5.0      |27      |
|Q5128     |ranibizumab  |4.07     |15      |
+----------+-------------+---------+--------+



In [0]:
# Financial summary
print("Average allowed amount by molecule and drug type:")
(
    spark.table(MED_TABLE)
    .groupBy("molecule", "drug_type")
    .agg(
        F.round(F.avg("allowed_amount"), 2).alias("avg_allowed"),
        F.round(F.avg("billed_amount"), 2).alias("avg_billed"),
        F.count("med_claim_id").alias("n_claims"),
    )
    .orderBy("molecule", "drug_type")
    .show(20, truncate=False)
)

Average allowed amount by molecule and drug type:
+-------------+---------------+-----------+----------+--------+
|molecule     |drug_type      |avg_allowed|avg_billed|n_claims|
+-------------+---------------+-----------+----------+--------+
|aflibercept  |biosimilar     |805.6      |1013.74   |20      |
|aflibercept  |originator     |1144.8     |1444.0    |78      |
|bevacizumab  |biosimilar     |568.62     |718.91    |66      |
|bevacizumab  |originator     |716.39     |916.08    |127     |
|infliximab   |biosimilar     |1730.07    |2193.7    |70      |
|infliximab   |originator     |3368.55    |4315.13   |46      |
|ocrelizumab  |new_formulation|10945.51   |13815.46  |25      |
|ocrelizumab  |originator     |11080.97   |14223.3   |81      |
|pembrolizumab|new_formulation|10119.35   |12827.46  |32      |
|pembrolizumab|originator     |10126.09   |12810.95  |81      |
|ranibizumab  |biosimilar     |82.08      |104.24    |50      |
|ranibizumab  |originator     |117.26     |148.27    |

In [0]:
# ICD-10 primary diagnosis distribution
print("Top ICD-10 primary diagnosis codes:")
(
    spark.table(MED_TABLE)
    .groupBy("icd10_primary")
    .count()
    .orderBy(F.desc("count"))
    .show(15, truncate=False)
)

Top ICD-10 primary diagnosis codes:
+-------------+-----+
|icd10_primary|count|
+-------------+-----+
|G35          |106  |
|C18.9        |76   |
|C34.90       |73   |
|H35.32       |64   |
|H35.354      |59   |
|C56.9        |47   |
|C64.9        |44   |
|H35.31       |40   |
|E11.319      |33   |
|K50.90       |32   |
|H34.8113     |28   |
|M45.9        |27   |
|K51.90       |26   |
|C67.9        |26   |
|M05.79       |24   |
+-------------+-----+
only showing top 15 rows


In [0]:
# Date distribution
print("Medical claims by month:")
(
    spark.table(MED_TABLE)
    .withColumn("month", F.month("service_date"))
    .groupBy("month")
    .agg(F.count("med_claim_id").alias("n_claims"))
    .orderBy("month")
    .show()
)

Medical claims by month:
+-----+--------+
|month|n_claims|
+-----+--------+
|   10|     247|
|   11|     252|
|   12|     253|
+-----+--------+



In [0]:
# Final row count
print("=" * 60)
print("NB03 output table - final row count")
print("=" * 60)
n = spark.table(MED_TABLE).count()
print(f"  {MED_TABLE:<55}: {n:>6,}")
print()
print(f"  of which seeded  : {spark.table(MED_TABLE).filter('is_seeded_duplicate = true').count():>6,}")
print(f"  of which clean   : {spark.table(MED_TABLE).filter('is_seeded_duplicate = false').count():>6,}")

NB03 output table - final row count
  cross_benefit_audit.medical_claims                     :    752

  of which seeded  :     72
  of which clean   :    680


---
## 9 - Summary

### Table written

| Table | Rows | Key fields |
|---|---|---|
| `medical_claims` | ~752 | med_claim_id, member_id, service_date, hcpcs_code, hcpcs_units, rendering_npi, is_seeded_duplicate |

### Seeded signal properties

Two detection signals are now coordinated between NB02 and NB03:

**NPI co-location** (`npi_match` in NB05)<br>
`seeded_duplicate_events.shared_prescriber_npi` is identical to `pharmacy_claims.prescriber_npi` (set by NB02) and to `medical_claims.rendering_npi` (set above). The buy-and-bill provider pool uses seeds 9001--9005, distinct from all other pools. A deterministic 20% of members also have their clean claims routed through this pool (via `is_buy_and_bill_member()`), producing realistic FP NPI matches that the risk scoring must distinguish from TP matches using the other signals.

**Dose concordance** (`dose_concordance_flag` in NB05)<br>
NB02 sets `pharmacy_claims.quantity_dispensed = adm_units * BILLUNITS * noise1`. NB03 sets `medical_claims.hcpcs_units = adm_units * noise2`. The concordance formula in NB05 computes `(qty / median_billunits) / med_units = noise1 / noise2 ≈ 1.0` for TPs. For FPs, `qty = BILLUNITSPKG ≈ 1` while `med_units` is drawn from the full `units_range` (40--70 for bevacizumab, etc.), so the FP concordance ratio is ≈ 0.03 -- well outside the 0.5--2.0 concordance band.

### What comes next

- **NB04** Two-pass matching engine: naive (member + HCPCS) and molecule bridge (member + molecule via crosswalk), 7-day window, confidence scoring.
- **NB05** Risk stratification: enriches NB04 matches with NPI co-location, dose concordance, dual-cost flag, and same-day flags; assigns HIGH/MEDIUM/LOW tiers.